# SIFT без автоматичної відмови за покриттям 95%

Перевіряємо **996 попередньо визначених кадрів**: 787 зі Scene_05 та 209 зі Scene_07. Це кадри, що пройшли геометричні перевірки завершеного аудиту. Відбір не залежить від успіху зіставлення.

Поріг 95% більше не зупиняє зіставлення. Сам показник покриття записується. Калібрування, попередня оцінка положення, діапазон висоти 20–120 м, масштаб, ресурсні ліміти та пороги SIFT залишаються чинними. Scene_21 не запускається: вона залишається для остаточної перевірки.

Реальні дані — тільки hosted Colab. Вручну скопіюйте оновлені `scripts/`, `requirements.txt` і цей notebook у Drive. Старий notebook аудиту та дослід 16 кадрів повторювати не потрібно. Нові результати зберігаються окремо.


In [7]:
import os
from pathlib import Path
if not os.environ.get('COLAB_RELEASE_TAG') or not Path('/content').is_dir():
    raise RuntimeError('Use a hosted Google Colab runtime, not a local runtime.')
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


## 1. Шляхи й код

Використайте ту саму Colab-сесію, якщо вона збереглася. Якщо потрібних залежностей немає, встановіть `INSTALL_DEPENDENCIES = True`. Версії NumPy/OpenCV мають збігатися з завершеним аудитом; їх буде показано нижче. Після зміни цих бібліотек може знадобитися перезапуск runtime.

`OUTPUT_ROOT` — нова спільна папка для обох робочих сцен. Не вказуйте папку старого аудиту або контрольного запуску 16 кадрів.


In [8]:
REPO_ROOT = Path('/content/drive/MyDrive/PhD/proj')
AUDIT_ROOT = REPO_ROOT / 'vml/anyvisloc-audit-v1'
ARCHIVE_ROOT = REPO_ROOT / 'AnyVisLoc'
OUTPUT_ROOT = REPO_ROOT / 'vml/anyvisloc-working-available-crop-v1'
STAGING_RECEIPTS = REPO_ROOT / 'vml/anyvisloc-working-staging-v1'

import sys, subprocess, importlib, json
INSTALL_DEPENDENCIES = False
if INSTALL_DEPENDENCIES:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(REPO_ROOT / 'requirements.txt')])
sys.path.insert(0, str(REPO_ROOT))
importlib.invalidate_caches()
prefix = 'scripts.pipeline.chance_constrained.visual_map_localization'
for name in ('anyvisloc', 'planar', 'smoke_localizer', 'anyvisloc_smoke', 'audit', 'working_campaign'):
    importlib.reload(importlib.import_module(prefix + '.' + name))
audit = importlib.import_module(prefix + '.audit')
campaign = importlib.import_module(prefix + '.working_campaign')
manifest = audit.read_json(AUDIT_ROOT / 'manifest.json')
import numpy as np, cv2
print('Required versions:', {'numpy': manifest['numpy'], 'opencv': manifest['opencv']})
print('Current versions:', {'numpy': np.__version__, 'opencv': cv2.__version__})
for scene_id, role in manifest['roles'].items():
    info = audit.read_json(AUDIT_ROOT / 'scenes' / scene_id / 'scene.json')
    print(scene_id, role, 'geometry candidates:', info['geometry_supported_count'])


Required versions: {'numpy': '2.1.3', 'opencv': '4.14.0'}
Current versions: {'numpy': '2.1.3', 'opencv': '4.14.0'}
Scene_05 working geometry candidates: 787
Scene_07 working geometry candidates: 209
Scene_21 holdout geometry candidates: 1082


## 2. Підготувати одну сцену

Спочатку залиште `SCENE_ID = 'Scene_05'`. Після завершення її зіставлення змініть на `'Scene_07'` та повторіть розділи 2–4. `OUTPUT_ROOT` залишайте однаковим: результати сцен зберігаються в різних підпапках.

Якщо ZIP є у Drive, він перевіряється за контрольною сумою з попереднього розпакування. Готові файли повторно використовуються, відсутні — розпаковуються. Якщо ZIP уже прибрали з Drive, але сцена залишилася на диску Colab, можна використати її: перед зіставленням програма перевірить контрольні суми потрібних вихідних файлів за аудитом.

Якщо немає ні архіву, ні розпакованої сцени, потрібен оригінальний архів цієї сцени. Автоматичного завантаження чи видалення немає. Архіви обробляємо послідовно з бюджетом до 5 GB у Drive.


In [11]:
SCENE_ID = 'Scene_07'  # Then change to Scene_07 and repeat sections 2-4.
STAGING_TAG = 'v1'
if manifest['roles'].get(SCENE_ID) != 'working':
    raise ValueError('Only frozen working scenes may be matched. Do not use Scene_21.')
DATASET_ROOT = Path('/content/anyvisloc-audit') / (SCENE_ID + '-' + STAGING_TAG)
archive_path = ARCHIVE_ROOT / (SCENE_ID + '.zip')
if archive_path.is_file():
    archive_bytes = sum(p.stat().st_size for p in ARCHIVE_ROOT.glob('*.zip'))
    if archive_bytes > 5_000_000_000:
        raise RuntimeError('Archive storage exceeds the 5 GB budget. Manage archives manually; no files were deleted.')
    receipts = [audit.read_json(p) for p in (AUDIT_ROOT / 'staging_receipts').glob(SCENE_ID + '-*.json')]
    hashes = {r['sha256'] for r in receipts}
    if len(hashes) != 1:
        raise ValueError('A unique original archive SHA-256 is required from the audit staging receipts.')
    print('Verifying archive and staging; this can take several minutes.')
    print(audit.stage_archive(archive_path, DATASET_ROOT, STAGING_RECEIPTS, hashes.pop()))
elif (DATASET_ROOT / SCENE_ID).is_dir():
    print('Using existing staging. Required source hashes will be verified before matching.')
else:
    raise FileNotFoundError('Upload the original scene ZIP or set STAGING_TAG to its existing runtime directory.')
print('Scene:', SCENE_ID, 'Dataset root:', DATASET_ROOT)


Using existing staging. Required source hashes will be verified before matching.
Scene: Scene_07 Dataset root: /content/anyvisloc-audit/Scene_07-v1


## 3. Виконати або продовжити зіставлення

Перед початком перевіряються джерела та відповідність завершеному аудиту. Використовуються ті самі збережені попередні оцінки положення. Покриття нижче 95% не є причиною відмови.

Прогрес друкується кожні 20 кадрів. Кожна завершена спроба має окремий файл із контрольною сумою. Після reconnect повторіть імпорти, підготовку потрібної сцени й цю клітинку: готові спроби не обчислюються повторно. Зміна коду, конфігурації або версій бібліотек потребує нової папки результатів.


In [12]:
summary = campaign.run_working_scene(
    DATASET_ROOT, AUDIT_ROOT, OUTPUT_ROOT, SCENE_ID,
    progress=lambda message: print(message, flush=True))
print('Campaign status:', summary['status'])
print('Completed:', summary['completed_trial_count'], '/', summary['expected_trial_count'])
print('Current scene:', summary['scenes'][SCENE_ID])


Scene_07: verifying source hashes for 209 frozen candidates
Scene_07: 20/209 trials saved or reused
Scene_07: 40/209 trials saved or reused
Scene_07: 60/209 trials saved or reused
Scene_07: 80/209 trials saved or reused
Scene_07: 100/209 trials saved or reused
Scene_07: 120/209 trials saved or reused
Scene_07: 140/209 trials saved or reused
Scene_07: 160/209 trials saved or reused
Scene_07: 180/209 trials saved or reused
Scene_07: 200/209 trials saved or reused
Scene_07: 209/209 trials saved or reused
Campaign status: completed
Completed: 996 / 996
Current scene: {'status': 'completed', 'expected_trial_count': 209, 'available_crop': {'trial_count': 209, 'accepted_count': 164, 'reject_count': 45, 'acc_at_m': {'1': 0.7033492822966507, '3': 0.7607655502392344, '5': 0.7703349282296651, '10': 0.7799043062200957, '20': 0.7799043062200957}, 'accepted_error_m': {'min': 0.010949173425593216, 'median': 0.222722471810884, 'mean': 0.5763241774336938, 'max': 20.339394912772597, 'rmse': 1.8518313433

## 4. Переглянути результати

Поки друга сцена не завершена, загальний статус `incomplete` очікуваний. Коли обидві завершені, буде `completed` і 996/996 спроб. Цю клітинку можна виконати після розриву сесії, щоб побачити збережений прогрес.

`legacy_95pct_counterfactual` означає **які з цих самих результатів залишилися б за старого порога**. Це обчислене порівняння, без другого запуску SIFT. Прийняті результати з похибками понад 1/3/5/10/20 м рахуються окремо: сам статус accepted не гарантує правильності.

Знаменник показника точності включає всі завершені спроби, зокрема відмови. Похибки вимірюються відносно еталонних координат датасету. Це ще не польотні ймовірності й не навчена модель похибок. Чорне заповнення карти залишається окремим невирішеним питанням даних; нових правил для нього цей дослід не вводить.


In [13]:
summary = campaign.summarize_campaign(AUDIT_ROOT, OUTPUT_ROOT)
print('Status:', summary['status'])
print('Completed:', summary['completed_trial_count'], '/', summary['expected_trial_count'])
print('Available crop:', summary['available_crop'])
print('Legacy 95% counterfactual:', summary['legacy_95pct_counterfactual'])
from IPython.display import display, Markdown
display(Markdown((OUTPUT_ROOT / 'report.md').read_text(encoding='utf-8')))


Status: completed
Completed: 996 / 996
Available crop: {'trial_count': 996, 'accepted_count': 395, 'reject_count': 601, 'acc_at_m': {'1': 0.29417670682730923, '3': 0.3433734939759036, '5': 0.35943775100401604, '10': 0.3755020080321285, '20': 0.38052208835341367}, 'accepted_error_m': {'min': 0.010949173425593216, 'median': 0.3185984465543652, 'mean': 6.058419644206704, 'max': 432.3123990139752, 'rmse': 37.835572112201845}, 'accepted_error_over_m': {'1': 102, '3': 53, '5': 37, '10': 21, '20': 16}}
Legacy 95% counterfactual: {'trial_count': 996, 'accepted_count': 241, 'reject_count': 755, 'acc_at_m': {'1': 0.21987951807228914, '3': 0.23493975903614459, '5': 0.23795180722891565, '10': 0.24096385542168675, '20': 0.24096385542168675}, 'accepted_error_m': {'min': 0.010949173425593216, 'median': 0.2315335835328543, 'mean': 0.5368133324056343, 'max': 20.339394912772597, 'rmse': 1.611867125143084}, 'accepted_error_over_m': {'1': 22, '3': 7, '5': 4, '10': 1, '20': 1}}


# Working-scene matching without a coverage threshold

Only the early 95% raster-extent coverage gate is disabled. Map coverage remains diagnostic.
SIFT thresholds, priors, AGL domain, scale and geometry/resource limits are unchanged.
The legacy policy column is a counterfactual applied to these same outcomes, not a second matcher run.
Accepted errors are relative to retained dataset GT; acceptance is not proof of correctness.

Campaign status: completed; completed 996/996 trials.

| Scene | Status | Completed/expected | Accepted | Accepted with error >1 m | Accepted with error >5 m |
| --- | --- | ---: | ---: | ---: | ---: |
| Scene_05 | completed | 787/787 | 231 | 85 | 34 |
| Scene_07 | completed | 209/209 | 164 | 17 | 3 |

## Aggregate results (completed trials only)

Available crop: {'trial_count': 996, 'accepted_count': 395, 'reject_count': 601, 'acc_at_m': {'1': 0.29417670682730923, '3': 0.3433734939759036, '5': 0.35943775100401604, '10': 0.3755020080321285, '20': 0.38052208835341367}, 'accepted_error_m': {'min': 0.010949173425593216, 'median': 0.3185984465543652, 'mean': 6.058419644206704, 'max': 432.3123990139752, 'rmse': 37.835572112201845}, 'accepted_error_over_m': {'1': 102, '3': 53, '5': 37, '10': 21, '20': 16}}

Legacy 95% gate counterfactual: {'trial_count': 996, 'accepted_count': 241, 'reject_count': 755, 'acc_at_m': {'1': 0.21987951807228914, '3': 0.23493975903614459, '5': 0.23795180722891565, '10': 0.24096385542168675, '20': 0.24096385542168675}, 'accepted_error_m': {'min': 0.010949173425593216, 'median': 0.2315335835328543, 'mean': 0.5368133324056343, 'max': 20.339394912772597, 'rmse': 1.611867125143084}, 'accepted_error_over_m': {'1': 22, '3': 7, '5': 4, '10': 1, '20': 1}}

Error cutoffs are descriptive accuracy levels, not fitted local/gross labels.
Both denominators include all completed candidate attempts, including rejects. Incomplete results are provisional.
Frame selection is fixed by the audit before matching. Excluded-domain frames are documented in the audit.
Black padding inside the raster is not removed or automatically treated as nodata. This limitation is unchanged.
Scene roles and original audit/control results are unchanged. No holdout matching or model fitting is performed.


## Що зберегти після обох сцен

Збережіть виконаний notebook із виводами. Для перевірки результатів завантажте всю нову папку `vml/anyvisloc-working-available-crop-v1` у локальний `results/vml/`.

Вона містить `run.json`, `summary.json`, `report.md`, а для кожної сцени — `status.json`, `summary.json`, `trials.csv` та підпапку `trials` із результатом кожного кадру. Папки старого аудиту й контрольного досліду 16 кадрів залишаються без змін.

Після Scene_05 змініть SCENE_ID на Scene_07 та повторіть розділи 2–4. Scene_21 не запускайте.
